In [ ]:
# This notebook performs NLP analysis on biotech/health articles, including:
# - Entity Recognition
# - Topic Modeling
# - Keyword Extraction
# - Text Classification
# - Trend Analysis

 
import pandas as pd
import numpy as np
import spacy
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from tqdm import tqdm

# Set style
plt.style.use('ggplot')
pd.set_option('display.max_colwidth', 200)
tqdm.pandas()

# Define the image save path
IMAGE_PATH = '/root/projects/portfolio_projects/data_analytics/civic_technology_project/data/images'

 
# Load data
data_file_path = '/root/projects/portfolio_projects/data_analytics/civic_technology_project/data/text_data.csv'
df = pd.read_csv(data_file_path)
print(f"Loaded {len(df)} articles")

# Clean text (lowercase and remove punctuation)
df['clean_text'] = df['story'].str.replace(r'[^\w\s]', '', regex=True).str.lower()


# 1. Entity Recognition (Simplified)

 
# Load medium-sized NLP model for faster processing
nlp = spacy.load("en_core_web_md")

def extract_entities_from_text(text):
    """Extract named entities from text using spaCy."""
    doc = nlp(text[:1000000])  # Limit to first 1M chars to prevent long processing
    return {
        'locations': [ent.text for ent in doc.ents if ent.label_ in ('GPE', 'LOC')],
        'organizations': [ent.text for ent in doc.ents if ent.label_ == 'ORG'],
        'people': [ent.text for ent in doc.ents if ent.label_ == 'PERSON'],
        'dates': [ent.text for ent in doc.ents if ent.label_ == 'DATE']
    }

# Apply entity extraction to the first 50 articles (can be adjusted for larger datasets)
df['entities'] = df['story'].head(50).progress_apply(extract_entities_from_text)

 
# Aggregate and visualize entities
entity_types = ['locations', 'organizations', 'people', 'dates']
plt.figure(figsize=(15, 10))

for i, entity_type in enumerate(entity_types, 1):
    plt.subplot(2, 2, i)
    all_entities = [ent for sublist in df['entities'].dropna() for ent in sublist[entity_type]]
    top_entities = Counter(all_entities).most_common(10)
    
    if top_entities:
        names, counts = zip(*top_entities)
        sns.barplot(x=list(counts), y=list(names), palette='viridis')
        plt.title(f'Top {entity_type.title()} Entities')
        plt.xlabel('Frequency')
        plt.ylabel(entity_type.title())

plt.tight_layout()
plt.savefig(f"{IMAGE_PATH}/entity_distribution.png")
plt.show()

# 2. Topic Modeling 

# Use TF-IDF vectorizer for better feature extraction
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df['clean_text'])

# Use NMF for topic modeling, which often works better than LDA for small datasets
nmf_model = NMF(n_components=5, random_state=42)
nmf_model.fit(tfidf_matrix)

 
# Display identified topics
def display_identified_topics(model, feature_names, num_top_words):
    """Display the topics identified by the NMF model."""
    for topic_idx, topic in enumerate(model.components_):
        print(f"\nTopic #{topic_idx}:")
        print(", ".join([feature_names[i] for i in topic.argsort()[:-num_top_words - 1:-1]]))

display_identified_topics(nmf_model, tfidf_vectorizer.get_feature_names_out(), 10)


# 3. Keyword Extraction (Alternative to Summarization)
 
def extract_top_keywords_from_text(text, num_keywords=10):
    """Extract top keywords using TF-IDF for a given text."""
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform([text])
    feature_array = np.array(vectorizer.get_feature_names_out())
    sorted_tfidf_indices = np.argsort(tfidf_matrix.toarray()).flatten()[::-1]
    return feature_array[sorted_tfidf_indices][:num_keywords]

# Apply to the first article
sample_text = df['story'].iloc[0]
print("Key Keywords:", ", ".join(extract_top_keywords_from_text(sample_text)))


# 4. Text Classification
# Define categories and corresponding keywords
category_keywords = {
    'therapy': ['therapy', 'treatment', 'drug', 'clinical'],
    'devices': ['device', 'wearable', 'sensor', 'implant'],
    'biotech': ['gene', 'cell', 'dna', 'mrna'],
    'digital': ['ai', 'digital', 'software', 'algorithm']
}

def categorize_article_based_on_keywords(text):
    """Assign category based on keywords found in the text."""
    text_lower = text.lower()
    for category, keywords in category_keywords.items():
        if any(keyword in text_lower for keyword in keywords):
            return category
    return 'other'

# Apply categorization to all articles
df['category'] = df['clean_text'].progress_apply(categorize_article_based_on_keywords)

 
# Visualize category distribution
plt.figure(figsize=(8, 5))
df['category'].value_counts().plot(kind='bar', color='teal')
plt.title('Document Category Distribution')
plt.ylabel('Number of Articles')
plt.xlabel('Category')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{IMAGE_PATH}/category_distribution.png")
plt.show()


# 5. Trend Analysis 

 
# Extract noun chunks (multi-word phrases) from articles as potential trends
def extract_noun_chunks_from_text(text):
    """Extract noun chunks from text to identify trends."""
    doc = nlp(text[:500000])  # Limit to 500K chars to reduce processing time
    return [chunk.text.lower() for chunk in doc.noun_chunks if len(chunk.text.split()) > 1]

# Apply noun chunk extraction
df['noun_phrases'] = df['story'].head(50).progress_apply(extract_noun_chunks_from_text)

 
# Find the most frequent trending phrases
all_noun_phrases = [phrase for sublist in df['noun_phrases'].dropna() for phrase in sublist]
trending_phrases = Counter(all_noun_phrases).most_common(20)

print("Top Trending Phrases:")
for phrase, count in trending_phrases:
    print(f"- {phrase} ({count} occurrences)")


# 6. Word Clouds by Category

 
# Generate word clouds for the top 3 categories
for category in df['category'].value_counts().index[:3]:
    category_text = ' '.join(df[df['category'] == category]['clean_text'])
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(category_text)
    
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(f'Word Cloud for {category.title()}')
    plt.axis('off')
    plt.savefig(f"{IMAGE_PATH}/wordcloud_{category}.png")
    plt.show()


# 7. Conclusions

# Generate a final report summarizing findings
top_organizations = Counter([org for sublist in df['entities'].dropna() for org in sublist['organizations']]).most_common(3)
top_people = Counter([person for sublist in df['entities'].dropna() for person in sublist['people']]).most_common(3)
top_locations = Counter([loc for sublist in df['entities'].dropna() for loc in sublist['locations']]).most_common(3)

print(f"""
Biotech & Health Analysis Report

#Key Findings:

1. Prominent Entities:
   - Organizations: {', '.join([org[0] for org in top_organizations])}
   - People: {', '.join([person[0] for person in top_people])}
   - Locations: {', '.join([loc[0] for loc in top_locations])}

2. Main Topics Identified:
   - Medical treatments and therapies
   - Technological devices and sensors
   - Biotechnology research
   - Digital health solutions

3. Category Distribution:
   - Most common category: {df['category'].value_counts().index[0]}
   - Emerging areas: {df['category'].value_counts().index[1]}, {df['category'].value_counts().index[2]}

4. Trending Phrases:
   - {trending_phrases[0][0]}
   - {trending_phrases[1][0]}
   - {trending_phrases[2][0]}

#Recommendations:
1. Focus research on {df['category'].value_counts().index[0]} technologies
2. Monitor developments from {top_organizations[0][0]} and {top_organizations[1][0]}
3. Explore applications in {trending_phrases[0][0]} and related areas
""")
